# Demo para la clase
Pensado para correrse en vivo: **configurá la celda 2 y usá *Run all***. Elegí modelos, dispositivo y de dónde sale el video; el notebook corre la acción `demo`
por `Toolkit` (la misma que `python -m src.cli <modelo> demo`, queda en la bitácora) y muestra el resultado acá mismo.

Qué se ve: el **video real a la izquierda** y lo que **predice el modelo a la derecha**; los frames de contexto (el modelo los ve) tienen borde gris y los que
imagina, borde naranja. Los modelos latentes (LeWM, V-JEPA 2, V-JEPA 2-AC) muestran su figura: decodificación, máscaras y curva de error.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

## 1. Configuración
Todo lo que se elige está en esta celda. Un modelo sin entrenar no rompe nada: aparece marcado y se explica cómo entrenarlo.

In [ ]:
# ============== ELEGÍ ACÁ ==============
PRUEBA = bool(os.environ.get("ML2_NB_LIMITE"))     # lo pone scripts/run_notebooks.py: modelos sin entrenar y clips cortos, para probar en CPU

MODELOS = ["genie_s", "convlstm_px", "gpt_s_kd"]   # 1 a 3 corridas; las latentes: "lewm", "vjepa2_s", "vjepa2_ac"
DEVICE = nbui.device_inicial()                      # el que detecta el repo: cuda, rocm, mps o cpu (dml y npu: ver inferencia.ipynb)
FUENTE = "ejemplo"      # "ejemplo" (el video de cámara de comma), "val" (un segmento de val) o "mp4" (un video propio)
SEGMENTO, INICIO = 0, 0           # con FUENTE = "val": qué segmento de val y desde qué frame
MP4 = "mi_video.mp4"              # con FUENTE = "mp4"
PASOS = 8                         # pasos de decodificación de genie y maskgit
# =======================================
if PRUEBA:
  MODELOS, PASOS, DEVICE = ["maskgit_s", "genie_s"], 1, "cpu"
CTX, HORIZON = (4, 2) if PRUEBA else (20, 60)       # 1 s de contexto real y 3 s imaginados (a 20 fps)
CLIP = dict(fuente=FUENTE, segmento=SEGMENTO, inicio=INICIO, mp4=MP4, device=DEVICE, pasos=PASOS, aleatorio=PRUEBA)
if PRUEBA:    # el modo prueba no deja nada en results/ ni en la bitácora del repo: todo va a una carpeta temporal
  import tempfile
  from src import rutas
  rutas.RESULTS = Path(tempfile.mkdtemp(prefix="ml2_prueba_")) / "results"

print("dispositivo:", DEVICE, "| fuente:", FUENTE, "| modo prueba (pesos sin entrenar):", PRUEBA)
nbui.backends();

### Modelos disponibles
*entrenada* = hay `checkpoints/<corrida>/final.pt` o está en `results/checkpoints.json`. Las demás se saltean con el comando que las entrena.

In [ ]:
nbui.catalogo_corridas(tuple(dict.fromkeys(MODELOS + ["genie_s", "gpt_s_kd", "convlstm_px", "lewm", "vjepa2_s", "vjepa2_ac"])));

## 2. La demo
Cada modelo sobre el clip elegido: real contra predicho, con su latencia por frame generado.

In [ ]:
for run in MODELOS[:1]:
  nbui.demo(run, ctx=CTX, horizon=HORIZON, frames=(CTX + HORIZON) * (1 if PRUEBA else 2), **CLIP)

## 3. Dos o tres modelos lado a lado
Mismo clip, mismos frames de contexto; abajo, la latencia por frame de cada uno (la de la demo incluye bajar el frame a la CPU).

In [ ]:
nbui.comparar(MODELOS[:3], ctx=CTX, horizon=HORIZON, frames=(CTX + HORIZON) * (1 if PRUEBA else 2), **CLIP);

## 4. Imaginar
El modelo **ve 1 s real e imagina 3 s**. En el video, el borde gris marca el contexto real y el naranja, lo imaginado (`imagina +k`, con k = frames a futuro).

In [ ]:
nbui.imaginar(MODELOS[0], ctx=CTX, horizon=HORIZON, **CLIP);

## 5. Si falta algo
- **Un modelo sin entrenar**: la tabla de la sección 1 dice el comando (`python -m src.cli <modelo> entrenar ...`); `nbui.lanzar(..., dry_run=False)` lo lanza desde acá, en segundo plano.
- **Otro dispositivo**: cambiá `DEVICE`; si no está disponible, la CLI lo avisa.
- **Un video propio**: `FUENTE = "mp4"` y la ruta en `MP4` (queda fuera de la distribución de entrenamiento).